# Lab 01 — Exploração e Limpeza de Dados com Pandas

> **Objetivo do lab:** transformar um CSV sujo em uma base minimamente confiável para análise.

Neste laboratório, vamos trabalhar com um arquivo de vendas de uma cafeteria.  
Ele foi escolhido porque tem problemas comuns em bases reais:

- valores ausentes;
- textos como `UNKNOWN` e `ERROR`;
- números salvos como texto;
- datas salvas como texto;
- colunas que precisam ser conferidas antes da análise.

**Importante:** neste primeiro lab, o foco não é fazer uma análise completa.  
O foco é aprender os comandos principais do Pandas para **explorar, diagnosticar e limpar dados**.

Ao final, você terá criado uma nova base chamada:

```text
cafe_sales_clean.csv
```

O dataset original será carregado diretamente do GitHub, então o aluno não precisa fazer upload manual do CSV.

Essa base limpa poderá ser usada no próximo lab de análise exploratória.

## Como usar este notebook

Rode as células **em ordem**, de cima para baixo.

Quando aparecer uma célula com comentário como:

```python
# escreva seu código abaixo
```

é a sua vez de tentar.

Logo depois de alguns exercícios, haverá uma célula de teste com `assert`.

### O que é `assert`?

O `assert` é uma forma simples de testar se uma resposta está correta.

Exemplo:

```python
assert resultado == 10
```

Se estiver correto, a célula roda sem mostrar erro.  
Se estiver errado, o Python mostra uma mensagem de erro.

## 1. Importando as bibliotecas

Antes de usar o Pandas, precisamos importar a biblioteca.

Vamos usar:

- **Pandas** para trabalhar com tabelas;
- **Matplotlib** para fazer um gráfico simples no final.

In [ ]:
# Importamos o Pandas com o apelido pd.
# Esse apelido é uma convenção usada por praticamente toda a comunidade Python.

import pandas as pd

# Importamos o Matplotlib para criar gráficos simples.
import matplotlib.pyplot as plt

## 2. Carregando o arquivo CSV direto do GitHub

Um arquivo CSV é uma tabela salva em formato de texto.

No Pandas, usamos:

```python
pd.read_csv()
```

Neste lab, vamos carregar o CSV diretamente de uma URL raw do GitHub.

Esse comando carrega o CSV e transforma os dados em um **DataFrame**.

Um **DataFrame** é uma tabela com linhas e colunas, parecida com uma planilha do Excel.

In [ ]:
# URL raw do dataset no GitHub.
# Assim o notebook funciona direto no Google Colab,
# sem precisar fazer upload manual do arquivo CSV.

url = "https://raw.githubusercontent.com/sidnei-almeida/mba-python-data-labs/refs/heads/master/data/dirty_cafe_sales.csv"

# Lendo o CSV com Pandas.
df = pd.read_csv(url)

# Mostrando as 5 primeiras linhas.
df.head()

## 3. Entendendo o tamanho da base

O atributo `.shape` mostra o tamanho do DataFrame.

Ele retorna dois números:

```python
(linhas, colunas)
```

**Exemplo:** se aparecer `(10000, 8)`, significa que a tabela tem **10.000 linhas** e **8 colunas**.

In [ ]:
# Quantas linhas e colunas existem no DataFrame?

df.shape

In [ ]:
# Também podemos separar essa informação em duas variáveis.
# Isso deixa o código mais legível.

linhas, colunas = df.shape

print("Quantidade de linhas:", linhas)
print("Quantidade de colunas:", colunas)

## 4. Conhecendo as colunas

O atributo `.columns` mostra os nomes das colunas.

Esse comando é muito usado no começo da análise, principalmente quando a base tem muitas colunas.

In [ ]:
# Listando os nomes das colunas.

df.columns

In [ ]:
# Transformando os nomes das colunas em uma lista do Python.

list(df.columns)

## 5. Primeiras e últimas linhas

Dois métodos muito usados:

- `.head()` mostra as primeiras linhas;
- `.tail()` mostra as últimas linhas.

Eles ajudam a ter uma primeira noção da estrutura da tabela.

In [ ]:
# Primeiras 5 linhas.

df.head()

In [ ]:
# Primeiras 10 linhas.

df.head(10)

In [ ]:
# Últimas 5 linhas.

df.tail()

## 6. Informações gerais com `.info()`

O método `.info()` é um dos comandos mais importantes para começar uma análise.

Ele mostra:

- quantidade de linhas;
- quantidade de colunas;
- nome das colunas;
- quantidade de valores não nulos;
- tipo de dado de cada coluna.

**Ponto de atenção:**  
Se uma coluna que deveria ser número aparece como `object`, provavelmente ela tem textos, erros ou valores misturados.

In [ ]:
# Informações gerais do DataFrame.

df.info()

### O que observar neste dataset?

Algumas colunas parecem numéricas, mas provavelmente aparecem como `object`:

- `Quantity`
- `Price Per Unit`
- `Total Spent`

Isso acontece porque existem valores como `UNKNOWN`, `ERROR` ou campos vazios misturados com os números.

Antes de analisar, precisamos limpar essas colunas.

## 7. Verificando valores ausentes

Valores ausentes são valores que não foram preenchidos.

No Pandas, usamos:

```python
df.isna().sum()
```

Esse comando conta quantos valores ausentes existem em cada coluna.

In [ ]:
# Contando valores ausentes por coluna.

df.isna().sum()

In [ ]:
# Calculando o percentual de valores ausentes por coluna.

percentual_ausentes = df.isna().mean() * 100

percentual_ausentes

## 8. Investigando valores únicos com `.value_counts()`

O método `.value_counts()` conta quantas vezes cada valor aparece em uma coluna.

Ele é muito útil para colunas categóricas, como:

- `Item`
- `Payment Method`
- `Location`

Também ajuda a encontrar valores estranhos.

In [ ]:
# Contagem dos itens vendidos.

df["Item"].value_counts()

In [ ]:
# Contagem dos métodos de pagamento.

df["Payment Method"].value_counts()

In [ ]:
# Contagem dos locais de consumo.

df["Location"].value_counts()

## 9. Procurando `UNKNOWN` e `ERROR`

Neste dataset, existem valores como:

- `UNKNOWN`
- `ERROR`

Eles não são considerados ausentes automaticamente pelo Pandas, mas para nossa análise eles indicam problema.

Vamos contar quantas vezes esses valores aparecem em cada coluna.

In [ ]:
# Lista com os valores problemáticos que queremos procurar.

valores_problematicos = ["UNKNOWN", "ERROR"]

# Para cada coluna, contamos quantas vezes aparecem UNKNOWN e ERROR.
# Se você ainda está começando em loops, foque primeiro no resultado.

for coluna in df.columns:
    quantidade = df[coluna].isin(valores_problematicos).sum()
    print(coluna, "->", quantidade)

## 10. Criando uma cópia antes da limpeza

Uma boa prática é **não alterar diretamente o DataFrame original**.

Vamos criar uma cópia chamada `df_limpo`.

A partir daqui, faremos as alterações nessa cópia.

In [ ]:
# Criando uma cópia do DataFrame original.

df_limpo = df.copy()

# Conferindo as primeiras linhas da cópia.

df_limpo.head()

## 11. Substituindo `UNKNOWN` e `ERROR` por valores ausentes

Para o Pandas entender que `UNKNOWN` e `ERROR` representam problemas, vamos substituir esses textos por `pd.NA`.

O `pd.NA` representa um valor ausente.

In [ ]:
# Substituindo UNKNOWN e ERROR por pd.NA.

df_limpo = df_limpo.replace(["UNKNOWN", "ERROR"], pd.NA)

# Conferindo novamente a quantidade de valores ausentes.

df_limpo.isna().sum()

## 12. Convertendo colunas numéricas

Agora vamos converter algumas colunas para número:

- `Quantity`
- `Price Per Unit`
- `Total Spent`

Para isso usamos:

```python
pd.to_numeric()
```

O parâmetro `errors="coerce"` significa:

> se encontrar algo que não pode ser convertido para número, transforme em valor ausente.

In [ ]:
# Convertendo Quantity para número.

df_limpo["Quantity"] = pd.to_numeric(df_limpo["Quantity"], errors="coerce")

# Convertendo Price Per Unit para número.

df_limpo["Price Per Unit"] = pd.to_numeric(df_limpo["Price Per Unit"], errors="coerce")

# Convertendo Total Spent para número.

df_limpo["Total Spent"] = pd.to_numeric(df_limpo["Total Spent"], errors="coerce")

# Verificando os tipos novamente.

df_limpo.info()

## 13. Convertendo a coluna de data

A coluna `Transaction Date` representa uma data.

Para converter texto em data, usamos:

```python
pd.to_datetime()
```

Assim o Pandas passa a entender essa coluna como data, não apenas texto.

In [ ]:
# Convertendo a coluna Transaction Date para data.

df_limpo["Transaction Date"] = pd.to_datetime(df_limpo["Transaction Date"], errors="coerce")

# Verificando o tipo da coluna.

df_limpo.info()

## 14. Verificando valores ausentes depois das conversões

Depois das conversões, é normal aparecerem mais valores ausentes.

Isso acontece porque valores inválidos foram transformados em:

- `NaN`: valor ausente numérico;
- `NaT`: valor ausente de data.

Essa etapa é importante porque mostra que a limpeza não é um único comando.  
A limpeza é um processo de **investigar, corrigir e conferir novamente**.

In [ ]:
# Valores ausentes depois das conversões.

df_limpo.isna().sum()

## 15. Criando uma coluna calculada

Agora que temos `Quantity` e `Price Per Unit` como números, podemos calcular o valor esperado da compra.

A lógica é:

```python
quantidade * preço_unitário
```

Vamos criar uma nova coluna chamada `Calculated Total`.

In [ ]:
# Criando uma nova coluna com o total calculado.

df_limpo["Calculated Total"] = df_limpo["Quantity"] * df_limpo["Price Per Unit"]

# Visualizando algumas colunas importantes.

df_limpo[["Quantity", "Price Per Unit", "Total Spent", "Calculated Total"]].head()

## 16. Comparando total informado e total calculado

A coluna `Total Spent` veio do arquivo original.

A coluna `Calculated Total` foi calculada por nós.

Comparar essas duas colunas ajuda a encontrar possíveis inconsistências.

In [ ]:
# Criando uma coluna que mostra a diferença entre o total informado e o total calculado.

df_limpo["Difference"] = df_limpo["Total Spent"] - df_limpo["Calculated Total"]

# Visualizando as primeiras linhas.

df_limpo[["Quantity", "Price Per Unit", "Total Spent", "Calculated Total", "Difference"]].head()

In [ ]:
# Verificando algumas estatísticas da diferença.

df_limpo["Difference"].describe()

## 17. Removendo linhas muito incompletas

Existem várias estratégias para lidar com dados ausentes.

Neste primeiro lab, vamos fazer uma escolha simples:

> remover linhas que não têm as informações essenciais para análise de vendas.

Para nossa análise inicial, vamos exigir valores em:

- `Item`
- `Quantity`
- `Price Per Unit`
- `Transaction Date`

**Observação:** em projetos reais, essa decisão deve ser pensada com cuidado.  
Aqui estamos simplificando para fins didáticos.

In [ ]:
# Colunas essenciais para nossa primeira análise.

colunas_essenciais = ["Item", "Quantity", "Price Per Unit", "Transaction Date"]

# Criando uma nova versão removendo linhas com valores ausentes nessas colunas.

df_limpo = df_limpo.dropna(subset=colunas_essenciais)

# Conferindo o novo tamanho da base.

df_limpo.shape

## 18. Preenchendo valores ausentes em colunas categóricas

Para colunas de texto, como `Payment Method` e `Location`, podemos preencher valores ausentes com `"Não informado"`.

Isso evita perder muitas linhas por causa de informações que não são essenciais para o cálculo de vendas.

In [ ]:
# Preenchendo valores ausentes em Payment Method.

df_limpo["Payment Method"] = df_limpo["Payment Method"].fillna("Não informado")

# Preenchendo valores ausentes em Location.

df_limpo["Location"] = df_limpo["Location"].fillna("Não informado")

# Conferindo valores ausentes novamente.

df_limpo.isna().sum()

## 19. Criando colunas de ano, mês e dia

Quando temos uma coluna de data, podemos extrair partes dela.

Vamos criar:

- `Year`
- `Month`
- `Day`

Essas colunas ajudam em análises por período.

In [ ]:
# Criando colunas a partir da data.

df_limpo["Year"] = df_limpo["Transaction Date"].dt.year
df_limpo["Month"] = df_limpo["Transaction Date"].dt.month
df_limpo["Day"] = df_limpo["Transaction Date"].dt.day

# Visualizando o resultado.

df_limpo[["Transaction Date", "Year", "Month", "Day"]].head()

## 20. Primeira análise simples depois da limpeza

Agora que os dados estão um pouco mais organizados, conseguimos fazer perguntas simples.

Por exemplo:

> **Quais produtos aparecem mais vezes?**

In [ ]:
# Contagem de vendas por item.

df_limpo["Item"].value_counts()

Também podemos calcular o faturamento por item usando `.groupby()`.

Não se preocupe se `groupby` ainda parecer estranho.  
Vamos estudar isso melhor em outro lab.

Por enquanto, pense assim:

> agrupar por item e somar o total calculado.

In [ ]:
# Faturamento por item.

faturamento_por_item = df_limpo.groupby("Item")["Calculated Total"].sum()

faturamento_por_item

## 21. Um gráfico simples com Pandas e Matplotlib

O Pandas consegue criar gráficos simples usando `.plot()`.

Vamos fazer um gráfico de barras com o faturamento por item.

**Não precisa decorar a parte visual agora.**  
O mais importante é entender que estamos usando uma coluna agrupada para criar um gráfico.

In [ ]:
# Ordenando do maior para o menor.

faturamento_por_item = faturamento_por_item.sort_values(ascending=False)

# Criando o gráfico.

faturamento_por_item.plot(kind="bar", figsize=(10, 5))

plt.title("Faturamento por Item")
plt.xlabel("Item")
plt.ylabel("Faturamento Calculado")
plt.xticks(rotation=45)
plt.show()

## 22. Salvando a base limpa

Agora vamos salvar a base limpa em um novo CSV.

No Google Colab, esse arquivo ficará salvo no ambiente temporário da sessão. Você pode baixá-lo pelo painel lateral de arquivos.

Esse arquivo poderá ser usado no próximo lab, focado em análise exploratória.

In [ ]:
# Salvando o DataFrame limpo em CSV.
# No Colab, este arquivo será salvo no ambiente temporário da sessão.
# Você pode baixá-lo pelo painel lateral de arquivos, se quiser.

df_limpo.to_csv("cafe_sales_clean.csv", index=False)

print("Arquivo cafe_sales_clean.csv salvo com sucesso!")

# Exercícios com correção simples

Agora é a sua vez.

A ideia aqui não é decorar tudo.  
A ideia é praticar os comandos principais com calma.

Em alguns exercícios, você deve salvar sua resposta em uma variável.  
Logo abaixo, haverá uma célula de teste com `assert`.

**Como saber se deu certo?**

- Se a célula de teste rodar sem erro, está correto.
- Se aparecer erro, volte na sua resposta e ajuste.

## Exercício 1 — Tamanho da base limpa

Salve o tamanho da base limpa na variável `tamanho_base_limpa`.

Dica: use `.shape`.

In [ ]:
# Exercício 1
# Salve o tamanho da base limpa na variável tamanho_base_limpa.

# escreva seu código abaixo

tamanho_base_limpa =

In [ ]:
# Teste do Exercício 1
# Rode esta célula depois de fazer o exercício.

assert tamanho_base_limpa == df_limpo.shape, "Confira se você usou df_limpo.shape."

print("Exercício 1 correto!")

## Exercício 2 — Primeiras linhas

Salve as 10 primeiras linhas de `df_limpo` na variável `primeiras_10_linhas`.

Dica: use `.head(10)`.

In [ ]:
# Exercício 2
# Salve as 10 primeiras linhas de df_limpo na variável primeiras_10_linhas.

# escreva seu código abaixo

primeiras_10_linhas =

In [ ]:
# Teste do Exercício 2

assert primeiras_10_linhas.shape[0] == 10, "Sua resposta precisa ter 10 linhas."
assert list(primeiras_10_linhas.columns) == list(df_limpo.columns), "As colunas devem ser as mesmas de df_limpo."

print("Exercício 2 correto!")

## Exercício 3 — Valores ausentes

Conte quantos valores ausentes ainda existem em cada coluna da base limpa.

Salve o resultado na variável `ausentes_por_coluna`.

Dica: use `.isna().sum()`.

In [ ]:
# Exercício 3
# Conte os valores ausentes em df_limpo.

# escreva seu código abaixo

ausentes_por_coluna =

In [ ]:
# Teste do Exercício 3

assert ausentes_por_coluna.equals(df_limpo.isna().sum()), "Confira se você usou df_limpo.isna().sum()."

print("Exercício 3 correto!")

## Exercício 4 — Produtos mais vendidos

Conte quantas vendas existem para cada item.

Salve o resultado na variável `contagem_itens`.

Dica: use `.value_counts()` na coluna `Item`.

In [ ]:
# Exercício 4
# Conte quantas vezes cada item aparece.

# escreva seu código abaixo

contagem_itens =

In [ ]:
# Teste do Exercício 4

assert contagem_itens.equals(df_limpo["Item"].value_counts()), "Confira se você usou value_counts() na coluna Item."
assert contagem_itens.sum() == len(df_limpo), "A soma das contagens deve ser igual ao número de linhas."

print("Exercício 4 correto!")

## Exercício 5 — Filtrando um produto

Crie um DataFrame chamado `vendas_coffee` contendo apenas as vendas de `Coffee`.

Dica:

```python
df_limpo[df_limpo["Item"] == "Coffee"]
```

In [ ]:
# Exercício 5
# Mostre apenas as vendas de Coffee.

# escreva seu código abaixo

vendas_coffee =

In [ ]:
# Teste do Exercício 5

assert len(vendas_coffee) > 0, "O filtro não deveria retornar vazio."
assert (vendas_coffee["Item"] == "Coffee").all(), "Todas as linhas devem ser do item Coffee."

print("Exercício 5 correto!")

## Exercício 6 — Vendas com quantidade maior que 3

Crie um DataFrame chamado `vendas_quantidade_maior_3`.

Ele deve conter apenas linhas em que `Quantity` seja maior que 3.

In [ ]:
# Exercício 6
# Mostre vendas com Quantity maior que 3.

# escreva seu código abaixo

vendas_quantidade_maior_3 =

In [ ]:
# Teste do Exercício 6

assert len(vendas_quantidade_maior_3) > 0, "O filtro não deveria retornar vazio."
assert (vendas_quantidade_maior_3["Quantity"] > 3).all(), "Todas as linhas devem ter Quantity maior que 3."

print("Exercício 6 correto!")

## Exercício 7 — Faturamento por método de pagamento

Agrupe os dados por `Payment Method` e some a coluna `Calculated Total`.

Salve o resultado na variável `faturamento_por_pagamento`.

Dica:

```python
df_limpo.groupby("Payment Method")["Calculated Total"].sum()
```

In [ ]:
# Exercício 7
# Calcule o faturamento por método de pagamento.

# escreva seu código abaixo

faturamento_por_pagamento =

In [ ]:
# Teste do Exercício 7

resposta_esperada = df_limpo.groupby("Payment Method")["Calculated Total"].sum()

assert faturamento_por_pagamento.equals(resposta_esperada), "Confira o groupby por Payment Method e a soma de Calculated Total."
assert faturamento_por_pagamento.sum() == df_limpo["Calculated Total"].sum(), "A soma total precisa bater com o faturamento total."

print("Exercício 7 correto!")

## Exercício 8 — Criando uma coluna simples

Crie uma nova coluna chamada `High Quantity`.

Ela deve receber:

- `True` quando `Quantity` for maior que 3;
- `False` nos demais casos.

Dica:

```python
df_limpo["High Quantity"] = df_limpo["Quantity"] > 3
```

In [ ]:
# Exercício 8
# Crie a coluna High Quantity.

# escreva seu código abaixo

In [ ]:
# Teste do Exercício 8

assert "High Quantity" in df_limpo.columns, "A coluna High Quantity precisa existir."
assert df_limpo["High Quantity"].equals(df_limpo["Quantity"] > 3), "A coluna deve indicar se Quantity é maior que 3."

print("Exercício 8 correto!")

## Desafio final — Item com maior faturamento

Descubra qual foi o item com maior faturamento total.

Salve o nome do item na variável `item_maior_faturamento`.

Dica: você pode usar:

- `.groupby()`
- `.sum()`
- `.sort_values()`
- `.idxmax()`

Uma forma possível é:

```python
faturamento_item = df_limpo.groupby("Item")["Calculated Total"].sum()
item_maior_faturamento = faturamento_item.idxmax()
```

In [ ]:
# Desafio final
# Descubra o item com maior faturamento total.

# escreva seu código abaixo

item_maior_faturamento =

In [ ]:
# Teste do Desafio Final

resposta_esperada = df_limpo.groupby("Item")["Calculated Total"].sum().idxmax()

assert item_maior_faturamento == resposta_esperada, "Confira se você encontrou o item com maior soma de Calculated Total."

print("Desafio final correto!")

# Fechamento

Neste lab, você praticou:

- **`pd.read_csv()`** para carregar CSV;
- **`.head()`** e **`.tail()`** para visualizar linhas;
- **`.shape`** para ver tamanho da base;
- **`.columns`** para ver nomes das colunas;
- **`.info()`** para entender tipos de dados;
- **`.isna().sum()`** para contar ausentes;
- **`.value_counts()`** para contar categorias;
- **`.replace()`** para substituir valores problemáticos;
- **`pd.to_numeric()`** para converter números;
- **`pd.to_datetime()`** para converter datas;
- **`.dropna()`** para remover linhas incompletas;
- **`.fillna()`** para preencher valores ausentes;
- criação de novas colunas;
- **`.groupby()`** para agrupar dados;
- **`.plot()`** para criar gráfico simples;
- **`.to_csv()`** para salvar a base limpa;
- **`assert`** para validar respostas.

No próximo lab, podemos usar a base `cafe_sales_clean.csv` para fazer uma análise exploratória mais completa.